# Travel Planner AI

## Part 1: Introduction



#### Project Background

Travel choices are fragmented and time-consuming. We build **Travel Planner AI**, a chatbot that combines conversational extraction with rule-based ranking.

#### Problem Statement

*Given hotel and itinerary datasets, build a chatbot that captures user requirements and returns top recommendations with itinerary.*

In [53]:
# Import the libraries
import sys
import subprocess
from IPython.display import display

def ensure_pandas_available():
    try:
        import pandas as pd  # type: ignore
        return pd
    except ModuleNotFoundError:
        in_venv = sys.prefix != getattr(sys, 'base_prefix', sys.prefix)

        install_commands = []
        if in_venv:
            install_commands.append([sys.executable, '-m', 'pip', 'install', 'pandas'])
        else:
            install_commands.append([sys.executable, '-m', 'pip', 'install', '--user', '--break-system-packages', 'pandas'])

        last_error = None
        for cmd in install_commands:
            try:
                subprocess.check_call(cmd)
                import pandas as pd  # type: ignore
                return pd
            except Exception as exc:
                last_error = exc

        raise RuntimeError(
            'pandas is not available and auto-install failed. '
            'Select a kernel with pandas (recommended: project .venv) or run manually: '
            f"{sys.executable} -m pip install --user --break-system-packages pandas"
        ) from last_error

pd = ensure_pandas_available()

hotels_df = pd.read_csv('data/hotels.csv')
itinerary_df = pd.read_csv('data/itineraries.csv')

print('Hotels dataset shape:', hotels_df.shape)
print('Itinerary dataset shape:', itinerary_df.shape)
display(hotels_df.head())


Hotels dataset shape: (9, 7)
Itinerary dataset shape: (9, 3)


,name,destination,price_per_night,style,rating,amenities,description
0,Seine View Inn,Paris,95,cultural,4.3,wifi|breakfast|work_friendly,Close to museums and metro with strong value p...
1,Louvre Grand Stay,Paris,210,cultural,4.6,wifi|breakfast|gym,Upscale central stay ideal for city exploration.
2,Montmartre Luxe Suites,Paris,320,relaxation,4.8,wifi|spa|pool|breakfast,Premium comfort with spa and neighborhood charm.
3,Tokyo Capsule Hub,Tokyo,80,adventure,4.1,wifi|work_friendly|airport_shuttle,Compact modern stay for budget explorers.
4,Shibuya Skyline Hotel,Tokyo,180,adventure,4.5,wifi|gym|breakfast,Well-connected property near major attractions.


#### Approach:

1. Conversation and requirement gathering
2. Intent-to-structure mapping
3. Rule-based ranking + itinerary generation

## Part 2: System Design

#### Dataset

- `data/hotels.csv`
- `data/itineraries.csv`

#### Workings of the Chatbot

The bot first captures destination, budget, and days, then personalizes recommendations and itinerary.

In [54]:
print("User: budget cultural trip to Paris for 3 days with wifi and breakfast")
print("Bot: Capturing preferences and returning top 3 recommendations...")

User: budget cultural trip to Paris for 3 days with wifi and breakfast
Bot: Capturing preferences and returning top 3 recommendations...


As shown above, the chatbot asks for missing fields, confirms captured preferences, and returns ranked options.

#### Building the Chatbot

We implement in 3 stages similar to ShopAssist:
- Stage 1: Intent capture + confirmation
- Stage 2: Mapping + ranking + validation
- Stage 3: Recommendation conversation

##### Major functions behind the Chatbot

- initialize_conversation
- extract_preferences_from_text
- intent_confirmation_layer
- dictionary_present
- compare_hotels_with_user
- recommendation_validation
- generate_itinerary
- dialogue_mgmt_system

## Part 3: Implementation

## Stage 1

### 3.1 - Import the libraries

In [55]:
import re
import ast
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

### API Key Setup (Optional if using OpenAI API)

Use your own key securely. Do **not** hardcode it in the notebook.


In [56]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")

print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))


OPENAI_API_KEY loaded: True


### Security Note

- Never hardcode API keys in notebooks or source files.
- Never commit secrets to GitHub.
- Use environment variables or secure secret managers.
- If a key is exposed, revoke/rotate it immediately.


In [57]:
hotels_df = pd.read_csv("data/hotels.csv")
itinerary_df = pd.read_csv("data/itineraries.csv")
destinations = sorted(hotels_df["destination"].unique().tolist())
print("Destinations:", destinations)

Destinations: ['Goa', 'Paris', 'Tokyo']


### 3.2 - Implementing Intent Clarity and Intent Confirmation Layers

### `initialize_conversation()`:

In [58]:
def initialize_conversation():
    return [{
        "role": "assistant",
        "content": "Hi, I am Travel Planner AI. Share destination, budget, and trip duration."
    }]

In [59]:
debug_conversation = initialize_conversation()
debug_conversation

[{'role': 'assistant',
  'content': 'Hi, I am Travel Planner AI. Share destination, budget, and trip duration.'}]

### `get_chat_completions()`:

In [60]:
def extract_preferences_from_text(user_text: str) -> dict:
    text = user_text.lower()
    pref = {
        "destination": None,
        "budget_tier": None,
        "trip_days": None,
        "style": None,
        "amenities": []
    }

    for d in destinations:
        if d.lower() in text:
            pref["destination"] = d
            break

    budget_map = {
        "budget": ["budget", "cheap", "affordable", "low cost"],
        "mid": ["mid", "moderate", "value"],
        "luxury": ["luxury", "premium", "high end", "5 star"]
    }
    for tier, kws in budget_map.items():
        if any(k in text for k in kws):
            pref["budget_tier"] = tier
            break

    for s in ["adventure", "cultural", "relaxation"]:
        if s in text:
            pref["style"] = s
            break

    for tok in re.split(r"[,\s]+", text):
        tok = tok.strip(".!? ")
        if tok in {"wifi", "breakfast", "pool", "gym", "spa", "airport_shuttle", "family_friendly", "work_friendly"}:
            pref["amenities"].append(tok)

    m = re.search(r"\b(\d{1,2})\s*days?\b", text)
    if m:
        val = int(m.group(1))
        if 1 <= val <= 14:
            pref["trip_days"] = val

    return pref


def get_chat_completions(conversation: list) -> str:
    user_msg = next((x["content"] for x in reversed(conversation) if x.get("role") == "user"), "")
    pref = extract_preferences_from_text(user_msg)
    missing = [k for k in ["destination", "budget_tier", "trip_days"] if not pref.get(k)]
    if missing:
        return "Need more info: " + ", ".join(missing)
    return "Great, I captured your intent. " + str(pref)


In [61]:
debug_user_input = "I need a budget cultural trip to Paris for 3 days with wifi and breakfast"
debug_conversation.append({"role": "user", "content": debug_user_input})
debug_response_assistant = get_chat_completions(debug_conversation)
print(debug_response_assistant)

Great, I captured your intent. {'destination': 'Paris', 'budget_tier': 'budget', 'trip_days': 3, 'style': 'cultural', 'amenities': ['wifi', 'breakfast']}


### `moderation_check()`:

In [62]:
def moderation_check(user_input: str):
    blocked = any(w in user_input.lower() for w in ["kill", "bomb", "terror", "harm"])
    return {"flagged": blocked}

print(moderation_check("I need a Tokyo trip"))
print(moderation_check("I want to kill"))

{'flagged': False}
{'flagged': True}


### `intent_confirmation_layer()`:

In [63]:
def intent_confirmation_layer(response_assistant: str):
    marker = "Great, I captured your intent. "
    if marker not in response_assistant:
        return {"category": "needs_clarification", "response": response_assistant}

    parsed = ast.literal_eval(response_assistant.split(marker, 1)[1])
    required = ["destination", "budget_tier", "trip_days"]
    if all(parsed.get(k) for k in required):
        return {"category": "travel_recommendation", "response": parsed}
    return {"category": "needs_clarification", "response": parsed}


debug_confirmation = intent_confirmation_layer(debug_response_assistant)
print(debug_confirmation)

{'category': 'travel_recommendation', 'response': {'destination': 'Paris', 'budget_tier': 'budget', 'trip_days': 3, 'style': 'cultural', 'amenities': ['wifi', 'breakfast']}}


### `dictionary_present()`:

In [64]:
def dictionary_present(response):
    return response if isinstance(response, dict) else None

response_dict_n = dictionary_present(debug_confirmation.get("response"))
print(type(response_dict_n), response_dict_n)

<class 'dict'> {'destination': 'Paris', 'budget_tier': 'budget', 'trip_days': 3, 'style': 'cultural', 'amenities': ['wifi', 'breakfast']}


## Stage 2

### 3.3 Implementing the Product Mapping and Information Extraction Layers

### `compare_hotels_with_user()`:

In [65]:
def budget_tier_from_price(price: int) -> str:
    if price < 120:
        return "budget"
    if price <= 220:
        return "mid"
    return "luxury"


def compare_hotels_with_user(user_requirements: dict, top_n: int = 3):
    rows = []
    for _, row in hotels_df.iterrows():
        score = 0.0
        if user_requirements.get("destination") == row["destination"]:
            score += 4
        if user_requirements.get("budget_tier") == budget_tier_from_price(int(row["price_per_night"])):
            score += 3
        if user_requirements.get("style") and user_requirements.get("style") == row["style"]:
            score += 2

        req_am = set(user_requirements.get("amenities") or [])
        row_am = set(str(row["amenities"]).split("|"))
        score += 1.5 * len(req_am & row_am)
        score += float(row["rating"]) / 2
        rows.append((score, row.to_dict()))

    rows.sort(key=lambda x: x[0], reverse=True)
    return rows[:top_n]


top_3_hotels = compare_hotels_with_user(response_dict_n)
for i, (s, h) in enumerate(top_3_hotels, 1):
    print(i, h["name"], "| score:", round(s, 2))

1 Seine View Inn | score: 14.15
2 Louvre Grand Stay | score: 11.3
3 Montmartre Luxe Suites | score: 9.4


### `product_validation_layer()`:

In [66]:
def recommendation_validation(hotel_recommendation):
    validated = []
    for score, hotel in hotel_recommendation:
        req = ["name", "destination", "price_per_night", "rating"]
        if all(k in hotel for k in req):
            validated.append((score, hotel))
    return validated

validated_data = recommendation_validation(top_3_hotels)
print("Validated recommendations:", len(validated_data))

Validated recommendations: 3


## Stage 3

### 3.4: Product Recommendation Layer

In [67]:
def generate_itinerary(destination: str, days: int, style: str | None):
    subset = itinerary_df[itinerary_df["destination"] == destination].sort_values("day")
    lines = []
    for _, r in subset.head(days).iterrows():
        msg = f"Day {int(r['day'])}: {r['plan']}"
        if style and style.lower() not in str(r["plan"]).lower():
            msg += f" (Style tip: add more {style} activities)"
        lines.append(msg)
    if not lines:
        lines = [f"Day {i}: Free exploration in {destination}" for i in range(1, days + 1)]
    return lines


for idx, (score, hotel) in enumerate(validated_data, 1):
    print(f"{idx}. {hotel['name']} | ₹{hotel['price_per_night']} | Rating {hotel['rating']} | Score {score:.2f}")

iti = generate_itinerary(
    destination=response_dict_n["destination"],
    days=int(response_dict_n["trip_days"]),
    style=response_dict_n.get("style")
)
print()
print("Suggested Itinerary:")
for line in iti:
    print("-", line)

1. Seine View Inn | ₹95 | Rating 4.3 | Score 14.15
2. Louvre Grand Stay | ₹210 | Rating 4.6 | Score 11.30
3. Montmartre Luxe Suites | ₹320 | Rating 4.8 | Score 9.40

Suggested Itinerary:
- Day 1: Visit the Louvre and evening Seine river walk for a cultural experience.
- Day 2: Explore Montmartre cafes and local art studios with light walking. (Style tip: add more cultural activities)
- Day 3: Day trip to Versailles and relaxed dinner cruise. (Style tip: add more cultural activities)


## Combining all the 3 stages

### 3.5 Dialogue Management System

In [68]:
def dialogue_mgmt_system(user_input: str):
    if not isinstance(user_input, str) or not user_input.strip():
        return {"status": "needs_clarification", "message": "Please share your destination, budget tier, and trip duration."}

    if moderation_check(user_input)["flagged"]:
        return {"status": "blocked", "message": "Input flagged by moderation."}

    convo = initialize_conversation()
    convo.append({"role": "user", "content": user_input})
    assistant_response = get_chat_completions(convo)
    confirmed = intent_confirmation_layer(assistant_response)

    if confirmed["category"] != "travel_recommendation":
        return {
            "status": "needs_clarification",
            "message": assistant_response,
            "suggestion": "Try: 'Budget cultural trip to Paris for 3 days with wifi and breakfast'."
        }

    user_dict = dictionary_present(confirmed["response"])
    if not user_dict:
        return {"status": "needs_clarification", "message": "Could not parse preferences. Please restate destination, budget tier, and days."}

    required = ["destination", "budget_tier", "trip_days"]
    missing = [k for k in required if not user_dict.get(k)]
    if missing:
        return {
            "status": "needs_clarification",
            "message": f"Missing required fields: {', '.join(missing)}",
            "captured_preferences": user_dict,
        }

    recos = recommendation_validation(compare_hotels_with_user(user_dict))
    itinerary = generate_itinerary(user_dict["destination"], int(user_dict["trip_days"]), user_dict.get("style"))

    return {
        "status": "ok",
        "captured_preferences": user_dict,
        "top_hotels": recos,
        "itinerary": itinerary,
    }


result = dialogue_mgmt_system("I need a budget cultural trip to Paris for 3 days with wifi and breakfast")
print(result["status"])


ok


In [69]:
def format_trip_plan(result_obj: dict) -> str:
    if result_obj.get("status") != "ok":
        msg = result_obj.get("message", "Unable to process request.")
        suggestion = result_obj.get("suggestion")
        return msg + (f"\nSuggestion: {suggestion}" if suggestion else "")

    pref = result_obj["captured_preferences"]
    lines = []
    lines.append("Trip Summary")
    lines.append(f"- Destination: {pref.get('destination')}")
    lines.append(f"- Budget Tier: {pref.get('budget_tier')}")
    lines.append(f"- Days: {pref.get('trip_days')}")
    if pref.get("style"):
        lines.append(f"- Style: {pref.get('style')}")

    lines.append("\nTop Recommendations:")
    for i, (score, hotel) in enumerate(result_obj["top_hotels"], 1):
        lines.append(
            f"{i}. {hotel['name']} ({hotel['destination']}) | ₹{hotel['price_per_night']}/night | "
            f"Rating {hotel['rating']} | Match Score {score:.2f}"
        )

    lines.append("\nItinerary:")
    for item in result_obj["itinerary"]:
        lines.append(f"- {item}")

    return "\n".join(lines)

print(format_trip_plan(result))


Trip Summary
- Destination: Paris
- Budget Tier: budget
- Days: 3
- Style: cultural

Top Recommendations:
1. Seine View Inn (Paris) | ₹95/night | Rating 4.3 | Match Score 14.15
2. Louvre Grand Stay (Paris) | ₹210/night | Rating 4.6 | Match Score 11.30
3. Montmartre Luxe Suites (Paris) | ₹320/night | Rating 4.8 | Match Score 9.40

Itinerary:
- Day 1: Visit the Louvre and evening Seine river walk for a cultural experience.
- Day 2: Explore Montmartre cafes and local art studios with light walking. (Style tip: add more cultural activities)
- Day 3: Day trip to Versailles and relaxed dinner cruise. (Style tip: add more cultural activities)


## Part V: Areas of Improvement and Final Comments

Current notebook prioritizes explainability and deterministic behavior.

Possible improvements:
1. Add real LLM API integration for richer language understanding
2. Add feedback loops for personalization
3. Add live pricing and weather-aware planning

### Future Scope of Work

In [70]:
print("Future scope: multilingual support, live pricing APIs, and adaptive itinerary optimization.")

Future scope: multilingual support, live pricing APIs, and adaptive itinerary optimization.


## Part VI: Function Calling API for Travel Planner

This section enhances the project with **function calling** concepts so the assistant can decide when to call travel-specific tools instead of only returning plain text.


### 6.1 What is Function Calling?

Function calling allows the model to return structured JSON arguments for a named function.
Your application then executes that function and can pass results back to the model.

In this travel planner, we use functions like:
- `findBestHotels(destination, budget_tier, top_n)`
- `convertCurrency(amount, from_currency, to_currency)`
- `createItinerary(destination, days, style)`


In [71]:
import json as pyjson

travel_functions = [
    {
        "name": "findBestHotels",
        "description": "Find the best hotels for a destination using budget and optional style preferences",
        "parameters": {
            "type": "object",
            "properties": {
                "destination": {"type": "string", "description": "Travel destination, e.g., Paris"},
                "budget_tier": {"type": "string", "enum": ["budget", "mid", "luxury"], "description": "Budget level"},
                "style": {"type": "string", "enum": ["adventure", "cultural", "relaxation"], "description": "Travel style"},
                "top_n": {"type": "integer", "description": "Number of recommendations", "default": 5}
            },
            "required": ["destination"]
        }
    },
    {
        "name": "convertCurrency",
        "description": "Convert currency values for trip budget estimation",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "number"},
                "from_currency": {"type": "string"},
                "to_currency": {"type": "string"}
            },
            "required": ["amount", "from_currency", "to_currency"]
        }
    },
    {
        "name": "createItinerary",
        "description": "Create a day-wise itinerary for a destination",
        "parameters": {
            "type": "object",
            "properties": {
                "destination": {"type": "string"},
                "days": {"type": "integer", "minimum": 1, "maximum": 14},
                "style": {"type": "string", "enum": ["adventure", "cultural", "relaxation"]}
            },
            "required": ["destination", "days"]
        }
    }
]

print('Registered function schemas:')
for fn in travel_functions:
    print('-', fn['name'])


Registered function schemas:
- findBestHotels
- convertCurrency
- createItinerary


### 6.2 Backend Implementations for Callable Functions


In [72]:
def findBestHotels(destination: str, budget_tier: str | None = None, style: str | None = None, top_n: int = 5):
    req = {
        "destination": destination,
        "budget_tier": budget_tier,
        "style": style,
        "amenities": []
    }
    ranked = compare_hotels_with_user(req, top_n=top_n)
    return [
        {
            "name": h["name"],
            "destination": h["destination"],
            "price_per_night": int(h["price_per_night"]),
            "rating": float(h["rating"]),
            "score": round(score, 2)
        }
        for score, h in ranked
    ]


def convertCurrency(amount: float, from_currency: str, to_currency: str):
    # Demo rates for notebook simulation
    rates = {
        ("USD", "INR"): 83.0,
        ("INR", "USD"): 1/83.0,
        ("EUR", "INR"): 90.0,
        ("INR", "EUR"): 1/90.0,
    }
    key = (from_currency.upper(), to_currency.upper())
    if key not in rates:
        return {"error": f"Rate not configured for {from_currency}->{to_currency}"}
    converted = amount * rates[key]
    return {
        "amount": amount,
        "from_currency": from_currency.upper(),
        "to_currency": to_currency.upper(),
        "converted_amount": round(converted, 2)
    }


def createItinerary(destination: str, days: int, style: str | None = None):
    return {"destination": destination, "days": days, "itinerary": generate_itinerary(destination, days, style)}


available_function_impl = {
    "findBestHotels": findBestHotels,
    "convertCurrency": convertCurrency,
    "createItinerary": createItinerary,
}

print('Callable implementations ready:', list(available_function_impl.keys()))


Callable implementations ready: ['findBestHotels', 'convertCurrency', 'createItinerary']


### 6.3 Simulated Model Function-Call Output

The API returns a `function_call` style payload. In production, this is produced by the model. For notebook reproducibility, we simulate that behavior below.


In [73]:
def simulate_model_function_call(user_query: str):
    q = user_query.lower()

    if "hotel" in q:
        dest = next((d for d in destinations if d.lower() in q), "Paris")
        budget = "budget" if "budget" in q else "luxury" if "luxury" in q else "mid"
        return {
            "index": 0,
            "message": {
                "role": "assistant",
                "content": None,
                "function_call": {
                    "name": "findBestHotels",
                    "arguments": pyjson.dumps({"destination": dest, "budget_tier": budget, "top_n": 5})
                }
            },
            "finish_reason": "function_call"
        }

    if "convert" in q and "usd" in q and "inr" in q:
        return {
            "index": 0,
            "message": {
                "role": "assistant",
                "content": None,
                "function_call": {
                    "name": "convertCurrency",
                    "arguments": pyjson.dumps({"amount": 100, "from_currency": "USD", "to_currency": "INR"})
                }
            },
            "finish_reason": "function_call"
        }

    if "itinerary" in q or "plan" in q:
        dest = next((d for d in destinations if d.lower() in q), "Tokyo")
        return {
            "index": 0,
            "message": {
                "role": "assistant",
                "content": None,
                "function_call": {
                    "name": "createItinerary",
                    "arguments": pyjson.dumps({"destination": dest, "days": 3, "style": "cultural"})
                }
            },
            "finish_reason": "function_call"
        }

    return {
        "index": 0,
        "message": {"role": "assistant", "content": "I can help with hotels, currency conversion, and itinerary planning."},
        "finish_reason": "stop"
    }


sample_response = simulate_model_function_call('Help me find the best hotels in Paris on a budget')
sample_response


{'index': 0,
 'message': {'role': 'assistant',
  'content': None,
  'function_call': {'name': 'findBestHotels',
   'arguments': '{"destination": "Paris", "budget_tier": "budget", "top_n": 5}'}},
 'finish_reason': 'function_call'}

### 6.4 Execute Function Calls from Model Output


In [74]:
def execute_function_call(response_obj: dict):
    message = response_obj.get("message", {})
    function_call = message.get("function_call")
    if not function_call:
        return {"type": "text", "content": message.get("content", "")}

    fn_name = function_call["name"]
    args = pyjson.loads(function_call.get("arguments", "{}"))

    if fn_name not in available_function_impl:
        return {"type": "error", "content": f"Unknown function: {fn_name}"}

    result = available_function_impl[fn_name](**args)
    return {"type": "function_result", "function": fn_name, "arguments": args, "result": result}


queries = [
    'Find best hotels in Paris for a budget trip',
    'Convert 100 USD to INR',
    'Create an itinerary for Tokyo'
]

for q in queries:
    print()
    print('User:', q)
    model_out = simulate_model_function_call(q)
    executed = execute_function_call(model_out)
    print('Function:', executed.get('function'))
    print('Result preview:', str(executed.get('result'))[:260], '...')



User: Find best hotels in Paris for a budget trip
Function: findBestHotels
Result preview: [{'name': 'Seine View Inn', 'destination': 'Paris', 'price_per_night': 95, 'rating': 4.3, 'score': 9.15}, {'name': 'Montmartre Luxe Suites', 'destination': 'Paris', 'price_per_night': 320, 'rating': 4.8, 'score': 6.4}, {'name': 'Louvre Grand Stay', 'destinatio ...

User: Convert 100 USD to INR
Function: convertCurrency
Result preview: {'amount': 100, 'from_currency': 'USD', 'to_currency': 'INR', 'converted_amount': 8300.0} ...

User: Create an itinerary for Tokyo
Function: createItinerary
Result preview: {'destination': 'Tokyo', 'days': 3, 'itinerary': ['Day 1: Shibuya crossing tour and street food tasting for an adventure-focused day. (Style tip: add more cultural activities)', 'Day 2: Temple trail in Asakusa with cultural workshops.', 'Day 3: Onsen session a ...


### 6.5 Function Calling Parameters (`none`, `auto`, specific function)

When using OpenAI Chat Completions with tools/functions:
- `none` → never call a function
- `auto` → model decides whether to call a function
- specific function name → force one function call

Below is an optional helper demonstrating the request shape (runs only if `openai` package and key are available).


In [75]:
def optional_openai_function_call_demo(user_query: str, mode: str = 'auto'):
    try:
        from openai import OpenAI
        import os
    except Exception:
        return {'status': 'skipped', 'reason': 'openai package not installed in current kernel'}

    if not __import__('os').getenv('OPENAI_API_KEY'):
        return {'status': 'skipped', 'reason': 'OPENAI_API_KEY not set'}

    client = OpenAI()

    # Note: API shape may differ by SDK version; adjust if needed.
    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': user_query}],
            functions=travel_functions,
            function_call=mode
        )
        msg = response.choices[0].message
        return {'status': 'ok', 'message': str(msg)}
    except Exception as e:
        return {'status': 'error', 'reason': str(e)}


demo_status = optional_openai_function_call_demo('Find the best hotels in Paris', mode='auto')
demo_status


{'status': 'skipped',
 'reason': 'openai package not installed in current kernel'}

## 6.6: Interactive Dialogue Demo

With function calling, this travel assistant can move from text-only guidance to tool-driven execution:
- pull hotel options from APIs,
- convert currencies in real time,
- create itinerary blocks from external data,
- and return structured, auditable outputs.

This pattern is the production-ready bridge between LLM reasoning and deterministic application logic.


### Interactive Dialogue Demo

Run the next cell to chat live with the Travel Planner. Type `exit` to stop.


In [77]:
def start_interactive_demo(max_turns: int = 50):
    print("Travel Planner Chat started (type 'exit' to stop)")
    turns = 0
    while turns < max_turns:
        user_input = input("\nYou: ").strip()
        if user_input.lower() in {"exit", "quit"}:
            print("Bot: Goodbye!")
            return
        result = dialogue_mgmt_system(user_input)
        print("\nBot:")
        print(format_trip_plan(result))
        turns += 1

    print("Bot: Session ended after max_turns. Run start_interactive_demo() again to continue.")

print("Interactive demo is ready. Run: start_interactive_demo()")


Interactive demo is ready. Run: start_interactive_demo()


In [78]:
start_interactive_demo()

Travel Planner Chat started (type 'exit' to stop)

Bot:
Please share your destination, budget tier, and trip duration.

Bot:
Need more info: budget_tier
Suggestion: Try: 'Budget cultural trip to Paris for 3 days with wifi and breakfast'.

Bot:
Please share your destination, budget tier, and trip duration.

Bot:
Trip Summary
- Destination: Paris
- Budget Tier: budget
- Days: 3
- Style: cultural

Top Recommendations:
1. Seine View Inn (Paris) | ₹95/night | Rating 4.3 | Match Score 14.15
2. Louvre Grand Stay (Paris) | ₹210/night | Rating 4.6 | Match Score 11.30
3. Montmartre Luxe Suites (Paris) | ₹320/night | Rating 4.8 | Match Score 9.40

Itinerary:
- Day 1: Visit the Louvre and evening Seine river walk for a cultural experience.
- Day 2: Explore Montmartre cafes and local art studios with light walking. (Style tip: add more cultural activities)
- Day 3: Day trip to Versailles and relaxed dinner cruise. (Style tip: add more cultural activities)
Bot: Goodbye!


## Part IV: Evaluation (Evidence Only)




### Evidence mapped to rubric

**System Design**
- Hybrid architecture implemented: intent extraction + rule-based ranking + itinerary generation
- Multi-stage workflow present (capture, validation, recommendation, orchestration)
- Real-world travel-planning use case with destination/budget/style constraints

**Technical Implementation**
- Modular functions implemented for parsing, scoring, validation, and dialogue management
- Function-calling schemas and execution router included (`findBestHotels`, `convertCurrency`, `createItinerary`)
- Robust handling for moderation checks and missing/invalid preference cases

**User Experience**
- Clear, structured response output (`format_trip_plan`)
- Clarification prompts and guided suggestion text when input is incomplete
- Consistent recommendation and itinerary presentation format

**Documentation**
- Notebook documents project goal, dataset, architecture, staged implementation, and function-calling design
- Security guidance for API key handling included
- Challenges/improvement notes retained for external review context
